# Atomic Scraper Service - Full API Test

This notebook tests all API endpoints of the service.

**Prerequisites:**
- Redis must be running (`redis-server`)
- Taskiq worker must be running (`uv run taskiq worker src.infrastructure.queue.broker:broker src.infrastructure.queue.session_actor`)
- FastAPI server must be running (`uv run uvicorn src.api.main:app --reload`)
- Playwright installed (`uv run playwright install --with-deps chromium`)

**Base URL:** `http://localhost:8000`
**API Key:** `default_internal_key`

In [1]:
import requests
import json
import time
import asyncio

In [2]:
# Helper function for JSON output (with truncation for large responses)
def show_response(response, label=None, max_len=2000):
    if label:
        print(f"\n{'='*50}")
        print(f"{label}")
        print('='*50)
    print(f"Status: {response.status_code}")
    try:
        data = response.json()
        text = json.dumps(data, indent=2)
        if len(text) > max_len:
            print(text[:max_len] + f"\n... [{len(text) - max_len} more chars]")
        else:
            print(text)
    except:
        print(response.text[:max_len])

In [3]:
BASE_URL = "http://localhost:8000"
API_KEY = "default_internal_key"
HEADERS = {"X-API-Key": API_KEY}

## 1. Health Check (No Auth)

In [4]:
response = requests.get(f"{BASE_URL}/healthz")
show_response(response, "Health Check")


Health Check
Status: 200
{
  "status": "degraded",
  "redis": "failed: Timeout connecting to server",
  "browser_pool": "degraded"
}


## 2. Scraper (Stateless)

In [5]:
# Full page scrape - requires Playwright
payload = {
    "url": "https://github.com/Unitech/pm2/issues/517",
    "wait_until": "domcontentloaded"
}
response = requests.post(
    f"{BASE_URL}/scraper",
    headers=HEADERS,
    json=payload,
    timeout=60
)
show_response(response, "Scraper")


Scraper
Status: 200
{
  "id": "b65af91a-37a3-49f5-b99f-3e5ac77a2500",
  "url": "https://github.com/Unitech/pm2/issues/517",
  "content": "<!DOCTYPE html><html lang=\"en\" data-color-mode=\"auto\" data-light-theme=\"light\" data-dark-theme=\"dark\" data-a11y-animated-images=\"system\" data-a11y-link-underlines=\"true\" class=\"js-focus-visible\" data-js-focus-visible=\"\"><head><style type=\"text/css\">.turbo-progress-bar {\n  position: fixed;\n  display: block;\n  top: 0;\n  left: 0;\n  height: 3px;\n  background: #0076ff;\n  z-index: 2147483647;\n  transition:\n    width 300ms ease-out,\n    opacity 150ms 150ms ease-in;\n  transform: translate3d(0, 0, 0);\n}\n</style>\n    <meta charset=\"utf-8\">\n  <link rel=\"dns-prefetch\" href=\"https://github.githubassets.com\">\n  <link rel=\"dns-prefetch\" href=\"https://avatars.githubusercontent.com\">\n  <link rel=\"dns-prefetch\" href=\"https://github-cloud.s3.amazonaws.com\">\n  <link rel=\"dns-prefetch\" href=\"https://user-images.github

In [6]:
response_content_d = json.loads(response.content.decode('utf-8'))
response_content_d['url']
# response_content_d['content']

'https://github.com/Unitech/pm2/issues/517'

## 3. Serper Search (Stateless)

In [5]:
# Google search (mocked for now)
payload = {
    "q": "artificial intelligence news"
}
response = requests.post(
    f"{BASE_URL}/serper",
    headers=HEADERS,
    json=payload
)
show_response(response, "Serper Search")


Serper Search
Status: 200
{
  "searchParameters": {
    "q": "artificial intelligence news",
    "type": "search",
    "engine": "google"
  },
  "organic": [
    {
      "title": "Example",
      "link": "https://example.com",
      "snippet": "Example snippet",
      "position": 1
    }
  ]
}


## 4. Omni-Parse (Stateless - AI Grounding)

In [ ]:
# AI element analysis - requires LLM server running
payload = {
    "base64_image": "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mNk+M9QDwADhgGAWjR9awAAAABJRU5ErkJggg==",
    "prompt": "Find all interactive elements"
}
response = requests.post(
    f"{BASE_URL}/omni-parse",
    headers=HEADERS,
    json=payload
)
show_response(response, "Omni-Parse")

## 5. Jina Extract (Stateless - AI Extraction)

In [8]:
# AI structured extraction - requires LLM server running
html_content = "<html><body><h1>Test Page</h1><p>Welcome to our site</p></body></html>"
payload = {
    "html": html_content,
    "extraction_schema": {
        "title": "string",
        "description": "string"
    }
}
response = requests.post(
    f"{BASE_URL}/jina-extract",
    headers=HEADERS,
    json=payload
)
show_response(response, "Jina Extract")


Jina Extract
Status: 200
{
  "extracted_data": {
    "raw_response": "\n\n# Extracted Structured Data\n\nBased on the HTML content analysis, here is the structured data:\n\n```json\n{\n  \"title\": \"Test Page\",\n  \"description\": \"Welcome to our site\"\n}\n```\n\n## Extraction Rationale:\n\n| Schema Field | Source Element | Reasoning |\n|--------------|----------------|-----------|\n| **title** | `<h1>Test Page</h1>` | The h1 tag conventionally represents the primary title of a web page, making it an ideal source for the 'title' field |\n| **description** | `<p>Welcome to our site</p>` | Paragraph text typically provides contextual information about the page content, suitable for the 'description' field |"
  }
}


In [7]:
# AI structured extraction - requires LLM server running
payload = {
    "html": response_content_d['content'],
    "extraction_schema": {
        "title": "string",
        "description": "string"
    }
}
response = requests.post(
    f"{BASE_URL}/jina-extract",
    headers=HEADERS,
    json=payload
)
show_response(response, "Jina Extract")


Jina Extract
Status: 500
Internal Server Error


In [24]:
response.content

b'{"extracted_data":{"raw_response":"```json\\n{\\n  \\"title\\": \\"Test Page\\",\\n  \\"description\\": \\"Welcome to our site\\"\\n}\\n```\\n"}}'

## 6. Yandex Maps Extract

In [ ]:
payload = {
    "category": "restaurants",
    "center": {
        "lat": 55.7558,
        "lng": 37.6173
    },
    "radius": 1000
}
response = requests.post(
    f"{BASE_URL}/api/v1/yandex-maps/extract",
    headers=HEADERS,
    json=payload
)
show_response(response, "Yandex Maps Extract")

## 7. Website Enrichment

In [ ]:
payload = {
    "url": "https://example.com",
    "crawl_about": True,
    "crawl_services": False
}
response = requests.post(
    f"{BASE_URL}/api/v1/enrich",
    headers=HEADERS,
    json=payload
)
show_response(response, "Website Enrichment")

## 8. Research Agent - Start Task

In [ ]:
payload = {
    "query": "What is the capital of France?",
    "mode": "speed"
}
response = requests.post(
    f"{BASE_URL}/api/v1/research/run",
    headers=HEADERS,
    json=payload
)
show_response(response, "Research Task Started")

# Save task_id for next tests
task_id = response.json().get("task_id") if response.status_code == 202 else None
print(f"\nTask ID: {task_id}")

## 9. Research Agent - Get Status

In [ ]:
if task_id:
    response = requests.get(
        f"{BASE_URL}/api/v1/research/status/{task_id}",
        headers=HEADERS
    )
    show_response(response, "Research Status")
else:
    print("No task_id available - skip this test")

## 10. Research Agent - Stream Progress

In [ ]:
if task_id:
    # Note: This is a generator, so we'll just check if it returns stream
    response = requests.get(
        f"{BASE_URL}/api/v1/research/stream/{task_id}",
        headers=HEADERS,
        stream=True
    )
    print(f"Status: {response.status_code}")
    print(f"Content-Type: {response.headers.get('content-type')}")
    print(f"First 200 chars: {response.text[:200] if response.text else 'empty'}")
else:
    print("No task_id available - skip this test")

## 11. Create Browser Session (Headless - Default)

In [8]:
# Default headless session
response = requests.post(
    f"{BASE_URL}/sessions",
    headers=HEADERS,
    json={}  # Uses defaults: headless=True, viewport=1280x720
)
show_response(response, "Create Session (Headless)")

session_id_headless = response.json().get("session_id") if response.status_code == 200 else None
print(f"\nSession ID (headless): {session_id_headless}")


Create Session (Headless)
Status: 500
Internal Server Error

Session ID (headless): None


## 12. Create Browser Session (Non-Headless / Visible)

In [7]:
# Non-headless session - browser window will be visible!
payload = {
    "headless": False,
    "viewport": {
        "width": 1280,
        "height": 720
    }
}
response = requests.post(
    f"{BASE_URL}/sessions",
    headers=HEADERS,
    json=payload
)
show_response(response, "Create Session (Non-Headless)")

session_id_visible = response.json().get("session_id") if response.status_code == 200 else None
print(f"\nSession ID (visible): {session_id_visible}")


Create Session (Non-Headless)
Status: 500
Internal Server Error

Session ID (visible): None


## 13. Execute DSL Command on Session

In [ ]:
if session_id_headless:
    # Test goto command
    command = {
        "type": "goto",
        "params": {
            "url": "https://example.com"
        }
    }
    response = requests.post(
        f"{BASE_URL}/sessions/{session_id_headless}/command",
        headers=HEADERS,
        json=command
    )
    show_response(response, "Execute Command (goto)")
else:
    print("No session_id available - skip this test")

## 14. Delete Session

In [ ]:
if session_id_headless:
    response = requests.delete(
        f"{BASE_URL}/sessions/{session_id_headless}",
        headers=HEADERS
    )
    show_response(response, "Delete Session (Headless)")

if session_id_visible:
    response = requests.delete(
        f"{BASE_URL}/sessions/{session_id_visible}",
        headers=HEADERS
    )
    show_response(response, "Delete Session (Visible)")

## 15. WebSocket Interactive Session (Manual Test)

**Note:** WebSocket testing requires a separate tool. Use this URL in a WebSocket client:

```
ws://localhost:8000/ws/{session_id}
```

Example session ID for testing: `test-session-123`

Commands to send via WebSocket:
```json
{"type": "goto", "params": {"url": "https://example.com"}}
{"type": "screenshot", "params": {}}
{"type": "scroll", "params": {"direction": "down"}}
```

In [ ]:
# WebSocket test using websockets library (optional)
# Uncomment to test:

# import websockets
# import asyncio

# async def test_ws():
#     # First create a session
#     session_resp = requests.post(f"{BASE_URL}/sessions", headers=HEADERS, json={"headless": False})
#     ws_session_id = session_resp.json().get("session_id")
#     print(f"Created session: {ws_session_id}")
    
#     # Connect to WebSocket
#     uri = f"ws://localhost:8000/ws/{ws_session_id}"
#     async with websockets.connect(uri) as ws:
#         # Send a command
#         await ws.send(json.dumps({"type": "goto", "params": {"url": "https://example.com"}}))
#         
#         # Receive response
#         response = await ws.recv()
#         print(f"Response: {response}")

# asyncio.run(test_ws())

## Summary

| # | Endpoint | Notes |
|---|----------|-------|
| 1 | `GET /healthz` | No auth |
| 2 | `POST /scraper` | Requires Playwright |
| 3 | `POST /serper` | Mocked (no API key) |
| 4 | `POST /omni-parse` | Requires LLM server |
| 5 | `POST /jina-extract` | Requires LLM server |
| 6 | `POST /api/v1/yandex-maps/extract` | Requires proxy |
| 7 | `POST /api/v1/enrich` | - |
| 8 | `POST /api/v1/research/run` | - |
| 9 | `GET /api/v1/research/status/{task_id}` | - |
| 10 | `GET /api/v1/research/stream/{task_id}` | SSE stream |
| 11 | `POST /sessions` (headless) | - |
| 12 | `POST /sessions` (non-headless) | Visible browser |
| 13 | `POST /sessions/{id}/command` | DSL commands |
| 14 | `DELETE /sessions/{id}` | - |
| 15 | WebSocket `/ws/{session_id}` | Manual test |